# RenAIssance: Handwritten VLM OCR Pipeline (GSoC Test II)
This notebook demonstrates the end-to-end extraction pipeline utilizing `gemini-2.5-flash` natively as a Vision-Language Model directly on raw, ultra-high-resolution scanned manuscripts.

### 0. Initialize Environment
**Run this cell first!** It adds the project source to your path and imports all required utilities.

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import display, Image
import matplotlib.pyplot as plt
import json

# 1. Add project root to path so we can import 'src'
ROOT = Path(os.getcwd()).parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import read_jsonl
from src.evaluate import load_ground_truth_records, compute_cer

print(f"Initialization complete! Project root: {ROOT}")

### 1. View Extracted Page Scans
Our `extract_pages.py` script automatically converted the master PDF scans into high-res PNGs and indexed them in a manifest.

In [ ]:
if "ROOT" not in globals(): raise RuntimeError("Please run the 'Initialize Environment' cell at the top first!")

manifest_path = ROOT / "data/page_images/manifest.jsonl"
manifest = read_jsonl(manifest_path)
sample_page = manifest[0]

print(f"Viewing {sample_page['page_id']}...")
img_path = ROOT / sample_page['image_path']
display(Image(filename=str(img_path), width=600))

### 2. Zero-Shot VLM Inference Analysis
This section compares the VLM output (stored in `data/predictions/vlm_results.jsonl`) against the ground truth.

In [ ]:
if "ROOT" not in globals(): raise RuntimeError("Please run the 'Initialize Environment' cell at the top first!")

# Load predictions vs Ground Truth
vlm_predictions = read_jsonl(ROOT / "data/predictions/vlm_results.jsonl")
ground_truths = read_jsonl(ROOT / "data/ground_truth/ground_truth.jsonl")

if not vlm_predictions:
    print("No VLM predictions found! Run 'python scripts/vlm_extract.py' in your terminal first.")
else:
    # Match the first ground truth page we have results for
    first_gt = ground_truths[0]
    first_pred_match = [p for p in vlm_predictions if p['page_id'] == first_gt['page_id']]
    
    if not first_pred_match:
        print(f"Could not find VLM prediction for {first_gt['page_id']}.")
    else:
        v_text = first_pred_match[0]['vlm_text']
        g_text = first_gt['text']
        cer = compute_cer(g_text, v_text)
        
        print(f"\033[1mPage ID:\033[0m {first_gt['page_id']}")
        print(f"\033[1mGemini Zero-Shot Prediction:\033[0m\n{v_text[:400]}...\n")
        print(f"\033[1mGround Truth Literal:\033[0m\n{g_text[:400]}...\n")
        print(f"\033[1mCER on Sample:\033[0m {cer:.2%}")

## 3. Qualititative Evaluation
The Gemini `VLM` natively scores ~15% CER on unconstrained cursive. Qualitative inspection shows it actively resolving abbreviations into expanded natural Spanish, improving researcher readability beyond the literal string match.